# AI Meeting Notes Analyzer using LangGraph

## Phase 1: Environment Setup

### Step 1.1: Verify Python Environment

**Project Objective:**  
Develop an AI-powered meeting notes analyzer using LangGraph and Large Language Models (LLMs).

The application will extract key topics, generate meeting summaries, identify action items and task owners, and classify task priorities.

**Technology Stack:**
- Python 3.12
- Jupyter Notebook (VS Code)
- LangGraph
- LangChain
- OpenAI LLM
- Pydantic
- Streamlit

**Purpose of this step:**

Verify the Python version and interpreter path before installing project dependencies.

This helps ensure that subsequent package installations use the correct Python environment.

In [1]:
# Step 1.1: Verify Python environment

import sys
import platform

print("=" * 55)
print("AI MEETING NOTES ANALYZER - ENVIRONMENT CHECK")
print("=" * 55)

print(f"Python Version    : {platform.python_version()}")
print(f"Python Executable : {sys.executable}")
print(f"Operating System  : {platform.system()}")
print(f"System Platform   : {platform.platform()}")

print("=" * 55)

if sys.version_info >= (3, 12):
    print("Status: Python 3.12+ detected.")
else:
    print("Status: Python version below recommended 3.12.")

AI MEETING NOTES ANALYZER - ENVIRONMENT CHECK
Python Version    : 3.12.0
Python Executable : c:\Users\nakul\DataScience\E&ICT_IIT_Guwahati\Capstone Project\CapstoneProject3_GenAIAgenticAI\AI-Meeting-Notes-Analyzer\myenv\python.exe
Operating System  : Windows
System Platform   : Windows-11-10.0.26300-SP0
Status: Python 3.12+ detected.


# Phase 2: Data Understanding and Exploration

## Step 2.1: Load the Meeting Transcript

**Objective:**

Load the provided meeting transcript into Python for further analysis.

**Dataset:** Meeting_Transcript.txt

**Activities:**
1. Locate the transcript file.
2. Read the file using UTF-8 encoding.
3. Store the transcript in a Python variable.
4. Display the transcript length and a preview.
5. Verify that the transcript was loaded successfully.

**Expected Outcome:**

The meeting transcript should be successfully loaded and ready for exploratory analysis and AI processing.

In [2]:
# Step 2.1: Load the meeting transcript

from pathlib import Path

# Define transcript file path
file_path = Path("Meeting_Transcript.txt")

# Validate file existence
if not file_path.is_file():
    raise FileNotFoundError(f"Transcript file not found: {file_path.resolve()}")

# Load transcript
meeting_transcript = file_path.read_text(encoding="utf-8-sig")

# Validate transcript content
if not meeting_transcript.strip():
    raise ValueError("The meeting transcript is empty.")

# Display results
print("=" * 55)
print("MEETING TRANSCRIPT - DATA LOADING")
print("=" * 55)

print(f"File Name        : {file_path.name}")
print(f"File Size        : {file_path.stat().st_size:,} bytes")
print(f"Character Count  : {len(meeting_transcript):,}")
print(f"Word Count       : {len(meeting_transcript.split()):,}")

print("\nTranscript Preview:")
print("-" * 55)
print(meeting_transcript[:1000])

print("\nStatus: Transcript loaded successfully.")

MEETING TRANSCRIPT - DATA LOADING
File Name        : Meeting_Transcript.txt
File Size        : 3,917 bytes
Character Count  : 3,825
Word Count       : 618

Transcript Preview:
-------------------------------------------------------
Product Manager: Good morning everyone. We have received several complaints about the mobile app crashing during login.

Customer Support Lead: Yes, our support team received more than 120 complaints in the last two days.

Product Manager: That is quite serious. We need to identify the root cause quickly.

Mobile Developer: I checked some of the crash logs yesterday and noticed that most of the issues are coming from Android users.

QA Tester: I also tried reproducing the issue on an Android device and the app crashed right after entering login credentials.

Backend Developer: Could the issue be related to the authentication API?

Mobile Developer: That is possible. The login request might be failing due to some recent backend changes.

Product Manager: When

## Step 2.2: Meeting Participants and Speaker Analysis

**Objective:**

Perform exploratory analysis of the meeting transcript to identify participants, speaker contributions, and transcript structure.

**Analysis Activities:**

1. Extract individual speaker statements.
2. Identify unique meeting participants.
3. Calculate the total number of statements.
4. Count statements contributed by each participant.
5. Calculate each participant's percentage contribution.
6. Identify any incorrectly formatted transcript lines.

**Methodology:**

Use Python's built-in regular expression and collection utilities to parse speaker names and their statements.

**Expected Outcome:**

A structured overview of meeting participants and their contributions, which will support accurate task ownership extraction in subsequent LangGraph agents.

In [3]:
# Step 2.2: Analyze meeting participants and speaker contributions

import re
from collections import Counter

# Initialize containers
speaker_statements = []
unparsed_lines = []

# Parse transcript line by line
for line_number, line in enumerate(meeting_transcript.splitlines(), start=1):

    line = line.strip()
    if not line:
        continue

    match = re.match(r"^([^:]+):\s*(.+)$", line)

    if match:
        speaker = match.group(1).strip()
        statement = match.group(2).strip()
        speaker_statements.append({"speaker": speaker, "statement": statement})
    else:
        unparsed_lines.append({"line_number": line_number, "text": line})

# Calculate speaker contributions
speaker_counts = Counter(item["speaker"] for item in speaker_statements)

total_statements = len(speaker_statements)
unique_speakers = len(speaker_counts)

# Display analysis
print("=" * 65)
print("MEETING PARTICIPANTS AND SPEAKER ANALYSIS")
print("=" * 65)

print(f"Total Statements       : {total_statements}")
print(f"Unique Participants    : {unique_speakers}")
print(f"Unparsed Lines         : {len(unparsed_lines)}")

print("\nSpeaker Contributions:")
print("-" * 65)

print(f"{'Speaker':<25} {'Statements':>12} {'Contribution %':>18}")
print("-" * 65)

for speaker, count in speaker_counts.most_common():

    contribution = (
        count / total_statements * 100
        if total_statements > 0 else 0
    )

    print(
        f"{speaker:<25} "
        f"{count:>12} "
        f"{contribution:>17.2f}%"
    )

print("-" * 65)

# Validate transcript structure
if unparsed_lines:
    print("\nWarning: Some lines could not be parsed.")

    for item in unparsed_lines[:5]:
        print(
            f"Line {item['line_number']}: "
            f"{item['text']}"
        )
else:
    print("\nStatus: All non-empty transcript lines parsed successfully.")

MEETING PARTICIPANTS AND SPEAKER ANALYSIS
Total Statements       : 47
Unique Participants    : 5
Unparsed Lines         : 0

Speaker Contributions:
-----------------------------------------------------------------
Speaker                     Statements     Contribution %
-----------------------------------------------------------------
Product Manager                     13             27.66%
QA Tester                           10             21.28%
Mobile Developer                     9             19.15%
Backend Developer                    9             19.15%
Customer Support Lead                6             12.77%
-----------------------------------------------------------------

Status: All non-empty transcript lines parsed successfully.


## Step 2.3: Transcript Quality Assessment and Preprocessing

**Objective:**

Clean and validate the meeting transcript while preserving the contextual information required for accurate LLM analysis.

**Preprocessing Strategy:**

1. Normalize inconsistent whitespace.
2. Remove unnecessary leading and trailing spaces.
3. Preserve speaker names and statement boundaries.
4. Preserve punctuation and sentence structure.
5. Preserve task commitments, deadlines, and urgency indicators.
6. Validate that preprocessing does not remove speaker statements.

**Important Considerations:**

Unlike traditional NLP pipelines, this GenAI application will not remove stopwords, perform stemming, or apply lemmatization.

These operations could remove contextual information necessary for meeting summarization, action item extraction, and priority classification.

**Expected Outcome:**

A clean, validated transcript suitable for processing by LangGraph agents without loss of meaningful information.

In [4]:
# Step 2.3: Clean and validate meeting transcript

import re

# Preserve original transcript
original_transcript = meeting_transcript

# Normalize each parsed statement
cleaned_statements = []

for item in speaker_statements:

    speaker = item["speaker"].strip()

    statement = re.sub(r"\s+", " ", item["statement"]).strip()

    cleaned_statements.append(f"{speaker}: {statement}")

# Reconstruct cleaned transcript
cleaned_transcript = "\n\n".join(cleaned_statements)

# Quality validation
original_count = len(speaker_statements)
cleaned_count = len(cleaned_statements)

assert original_count == cleaned_count, ("Statement count mismatch after preprocessing.")

assert cleaned_transcript.strip(), ("Cleaned transcript is empty.")

# Display preprocessing results
print("=" * 60)
print("TRANSCRIPT QUALITY AND PREPROCESSING REPORT")
print("=" * 60)

print(f"Original Characters : {len(original_transcript):,}")
print(f"Cleaned Characters  : {len(cleaned_transcript):,}")
print(f"Original Statements : {original_count}")
print(f"Cleaned Statements  : {cleaned_count}")
print(f"Unique Participants : {unique_speakers}")
print(f"Unparsed Lines      : {len(unparsed_lines)}")

print("\nCleaned Transcript Preview:")
print("-" * 60)
print(cleaned_transcript[:800])

print("\n" + "=" * 60)
print("Status: Preprocessing completed successfully.")
print("All parsed speaker statements preserved.")

TRANSCRIPT QUALITY AND PREPROCESSING REPORT
Original Characters : 3,825
Cleaned Characters  : 3,825
Original Statements : 47
Cleaned Statements  : 47
Unique Participants : 5
Unparsed Lines      : 0

Cleaned Transcript Preview:
------------------------------------------------------------
Product Manager: Good morning everyone. We have received several complaints about the mobile app crashing during login.

Customer Support Lead: Yes, our support team received more than 120 complaints in the last two days.

Product Manager: That is quite serious. We need to identify the root cause quickly.

Mobile Developer: I checked some of the crash logs yesterday and noticed that most of the issues are coming from Android users.

QA Tester: I also tried reproducing the issue on an Android device and the app crashed right after entering login credentials.

Backend Developer: Could the issue be related to the authentication API?

Mobile Developer: That is possible. The login request might be failing du

## Step 3.3: Secure OpenAI API Key Configuration

**Objective:**

Securely configure the OpenAI API key using environment variables.

**Approach:**

1. Store the OpenAI API key in a `.env` file.
2. Load environment variables using `python-dotenv`.
3. Validate that the API key is available.
4. Avoid printing or exposing the actual API key.
5. Exclude sensitive files from GitHub using `.gitignore`.

**Security Best Practices:**

- Never hardcode API keys in Python scripts or notebooks.
- Never upload `.env` files to GitHub.
- Never display API keys in notebook outputs.
- Use deployment platform secrets for production applications.

**Expected Outcome:**

The OpenAI API key should load successfully and be available for subsequent LLM initialization.

In [ ]:
# Step 3.3: Securely load OpenAI API key

import os
from pathlib import Path
from dotenv import load_dotenv

# Define environment file path
env_path = Path(".env")

# Validate .env file existence
if not env_path.is_file():
    raise FileNotFoundError(f".env file not found: {env_path.resolve()}")

# Load environment variables
load_dotenv(dotenv_path=env_path, override=True)

# Retrieve API key securely
openai_api_key = os.getenv("OPENAI_API_KEY")

# Validate API key
if not openai_api_key or not openai_api_key.strip():
    raise ValueError("OPENAI_API_KEY is missing or empty in the .env file.")

# Display safe configuration status
print("=" * 60)
print("OPENAI API KEY CONFIGURATION")
print("=" * 60)

print(f".env File Found       : {env_path.is_file()}")
print(f"API Key Configured   : {bool(openai_api_key)}")
print("API Key Visibility   : Hidden for security")

print("=" * 60)
print("Status: OpenAI API key loaded successfully.")

OPENAI API KEY CONFIGURATION
.env File Found       : True
API Key Configured   : True
API Key Visibility   : Hidden for security
Status: OpenAI API key loaded successfully.


## Step 3.4: Initialize and Test the OpenAI LLM

**Objective:**

Initialize the OpenAI Large Language Model using LangChain and verify successful communication with the OpenAI API.

**Model Configuration:**

- Model: gpt-4o-mini
- Temperature: 0
- Maximum Output Tokens: 150
- Request Timeout: 30 seconds
- Maximum Retries: 2

**Implementation Steps:**

1. Import ChatOpenAI from LangChain.
2. Initialize the language model.
3. Submit a simple test prompt.
4. Retrieve the model response.
5. Display token usage and response metadata, when available.
6. Verify successful API communication.

**Expected Outcome:**

The LLM should respond successfully to a test prompt, confirming that the application can communicate with OpenAI.

**Note:**

This step tests API connectivity only. Meeting transcript analysis and LangGraph agent development will be implemented in subsequent steps.

In [8]:
# Step 3.4: Initialize and test OpenAI LLM

from langchain_openai import ChatOpenAI

os.environ.pop("SSL_CERT_FILE", None)

# Initialize the language model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=150,
    timeout=30,
    max_retries=2
)

# Define a simple test prompt
test_prompt = ("In two sentences, explain how Generative AI can help organizations analyze meeting transcripts.")

# Invoke the LLM
response = llm.invoke(test_prompt)

# Display results
print("=" * 60)
print("OPENAI LLM CONNECTION TEST")
print("=" * 60)

print("\nModel Response:")
print("-" * 60)
print(response.content)

print("\nToken Usage:")
print("-" * 60)

usage = response.usage_metadata or {}

print(f"Input Tokens  : {usage.get('input_tokens', 'N/A')}")
print(f"Output Tokens : {usage.get('output_tokens', 'N/A')}")
print(f"Total Tokens  : {usage.get('total_tokens', 'N/A')}")

print("\nModel Information:")
print("-" * 60)
print(f"Model Configured : {llm.model_name}")

print("=" * 60)
print("Status: LLM API request completed successfully.")

OPENAI LLM CONNECTION TEST

Model Response:
------------------------------------------------------------
Generative AI can process and summarize meeting transcripts, extracting key insights and action items to enhance decision-making and accountability. Additionally, it can identify patterns and trends in discussions, enabling organizations to improve future meetings and collaboration strategies.

Token Usage:
------------------------------------------------------------
Input Tokens  : 23
Output Tokens : 44
Total Tokens  : 67

Model Information:
------------------------------------------------------------
Model Configured : gpt-4o-mini
Status: LLM API request completed successfully.


## Step 3.5: Baseline Meeting Transcript Analysis

**Objective:**

Evaluate the ability of a single Large Language Model to analyze the meeting transcript before implementing the LangGraph multi-agent workflow.

**Analysis Requirements:**

1. Generate a concise meeting summary (3–5 sentences).
2. Identify the key discussion topics.
3. Extract actionable tasks.
4. Identify task owners.
5. Classify task priorities as High, Medium, or Low.

**Methodology:**

Use a single LLM invocation with a structured prompt.

The model must rely only on information available in the transcript and avoid inventing task owners, deadlines, or decisions.

**Evaluation Strategy:**

This response will serve as the initial baseline for comparison with the LangGraph multi-agent system.

Later evaluations will examine:

- Action item extraction completeness
- Task ownership accuracy
- Priority classification consistency
- Summary faithfulness
- Hallucinations and unsupported information
- Token usage and API latency

**Expected Outcome:**

A structured meeting analysis generated directly by the LLM.

In [9]:
# Step 3.5: Baseline meeting transcript analysis

import os
import time
from langchain_core.messages import SystemMessage, HumanMessage

# Remove invalid SSL certificate file override, if present
os.environ.pop("SSL_CERT_FILE", None)

# Define baseline instructions
system_prompt = """
You are an experienced AI Meeting Notes Analyst.

Analyze the provided meeting transcript and generate:

1. Meeting Summary (3-5 sentences)
2. Key Discussion Topics
3. Action Items
4. Task Owners
5. Task Priorities (High, Medium, or Low)

Rules:
- Use only information explicitly supported by the transcript.
- Do not invent tasks, owners, deadlines, or decisions.
- Preserve the speaker roles exactly as provided.
- Label missing owners as "Not specified".
- Distinguish confirmed facts from suspected causes.
- Identify deadlines only when mentioned.
- Assign priority based on urgency and context.
- If no action items exist, state that clearly.
- Present the results in readable Markdown.
"""

# Prepare messages
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=cleaned_transcript)
]

# Record execution time
start_time = time.perf_counter()

# Invoke baseline LLM
baseline_response = llm.invoke(messages)

execution_time = time.perf_counter() - start_time

# Preserve results for later evaluation
baseline_report = baseline_response.content
baseline_usage = baseline_response.usage_metadata or {}

# Display results
print("=" * 70)
print("BASELINE MEETING TRANSCRIPT ANALYSIS")
print("=" * 70)

print("\nGENERATED MEETING REPORT")
print("-" * 70)
print(baseline_report)

print("\n" + "=" * 70)
print("BASELINE PERFORMANCE METRICS")
print("=" * 70)

print(f"Execution Time : {execution_time:.2f} seconds")
print(f"Input Tokens   : {baseline_usage.get('input_tokens', 'N/A')}")
print(f"Output Tokens  : {baseline_usage.get('output_tokens', 'N/A')}")
print(f"Total Tokens   : {baseline_usage.get('total_tokens', 'N/A')}")

print("=" * 70)
print("Status: Baseline analysis completed successfully.")

BASELINE MEETING TRANSCRIPT ANALYSIS

GENERATED MEETING REPORT
----------------------------------------------------------------------
# Meeting Summary
The team discussed a critical issue regarding the mobile app crashing during login, which has resulted in over 120 complaints from users in the last two days. The Mobile Developer and QA Tester confirmed that the crashes are primarily affecting Android users and may be linked to recent changes in the authentication API. Action items were assigned to investigate the root cause, implement fixes, and communicate with users about the ongoing efforts to resolve the issue.

# Key Discussion Topics
- Complaints about the mobile app crashing during login
- Identification of the root cause of the crashes
- Recent changes to the authentication API
- Urgency of resolving the issue to prevent user frustration
- Testing on different Android versions

# Action Items
1. Review the Android login

BASELINE PERFORMANCE METRICS
Execution Time : 3.10 secon

## Step 3.6: Optimize LLM Output Configuration

**Objective:**

Resolve baseline response truncation by increasing the maximum output-token limit.

**Issue Identified:**

The initial baseline analysis generated exactly 150 output tokens, matching the configured maximum token limit.

As a result, the meeting report was incomplete.

**Optimization Strategy:**

1. Increase the maximum output tokens from 150 to 2,000.
2. Maintain the same LLM model and temperature.
3. Reuse the original system prompt and meeting transcript.
4. Generate a complete baseline meeting report.
5. Capture token usage and execution time.
6. Check the model's reported completion status.

**Evaluation Criteria:**

- Meeting summary contains 3–5 sentences.
- Key discussion topics are identified.
- Action items are extracted.
- Task owners are correctly associated.
- Task priorities are classified.
- No unsupported information is introduced.

**Expected Outcome:**

A complete baseline meeting report suitable for comparison with the upcoming LangGraph multi-agent workflow.

In [10]:
# Step 3.6: Optimize LLM output and regenerate baseline report

import os
import time
from langchain_openai import ChatOpenAI

# Apply previously identified SSL environment fix
os.environ.pop("SSL_CERT_FILE", None)

# Initialize LLM with increased output capacity
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=2000,
    timeout=60,
    max_retries=2
)

# Measure execution time
start_time = time.perf_counter()

# Reuse the original baseline messages
baseline_response = llm.invoke(messages)

execution_time = time.perf_counter() - start_time

# Store results for future comparison
baseline_report = baseline_response.content
baseline_usage = baseline_response.usage_metadata or {}

# Retrieve completion status
finish_reason = baseline_response.response_metadata.get("finish_reason", "Unknown")

# Display generated report
print("=" * 70)
print("OPTIMIZED BASELINE MEETING ANALYSIS")
print("=" * 70)

print("\nGENERATED MEETING REPORT")
print("-" * 70)
print(baseline_report)

print("\n" + "=" * 70)
print("BASELINE PERFORMANCE METRICS")
print("=" * 70)

print(f"Execution Time : {execution_time:.2f} seconds")
print(f"Input Tokens   : {baseline_usage.get('input_tokens', 'N/A')}")
print(f"Output Tokens  : {baseline_usage.get('output_tokens', 'N/A')}")
print(f"Total Tokens   : {baseline_usage.get('total_tokens', 'N/A')}")
print(f"Finish Reason  : {finish_reason}")

print("=" * 70)

if finish_reason == "stop":
    print("Status: Model completed its response normally.")
elif finish_reason == "length":
    print("Warning: Response reached the output-token limit.")
else:
    print("Status: Review the finish reason before evaluation.")

OPTIMIZED BASELINE MEETING ANALYSIS

GENERATED MEETING REPORT
----------------------------------------------------------------------
# Meeting Summary
The team discussed a critical issue regarding the mobile app crashing during login, which has resulted in over 120 complaints from users in the last two days. The Mobile Developer and QA Tester confirmed that the crashes are primarily affecting Android users and may be linked to recent changes in the authentication API. Action items were assigned to investigate the root cause, implement fixes, and communicate with users about the ongoing efforts to resolve the issue.

# Key Discussion Topics
- Complaints about the mobile app crashing during login
- Identification of the root cause of the crashes
- Potential link to recent backend changes in the authentication API
- Urgency of resolving the issue to prevent user frustration
- Testing on different Android versions

# Action Items
1. Review the Android login module for error handling.
2. Pr

# Phase 4: LangGraph Multi-Agent Architecture

## Step 4.1: Define Structured Output Schemas

**Objective:**

Define validated data structures for the outputs generated by the specialized AI agents.

**Why Structured Outputs Matter:**

LLM responses may vary in formatting and completeness.

Pydantic models provide consistent schemas that can be validated before information is passed between LangGraph agents.

**Planned Schemas:**

1. TopicExtractionOutput
   - Key discussion topics.

2. MeetingSummaryOutput
   - Concise meeting summary.

3. ActionItem
   - Task description.
   - Task owner.
   - Deadline.
   - Supporting transcript evidence.

4. ActionExtractionOutput
   - Collection of extracted action items.

5. PriorityClassificationOutput
   - Priority assigned to each task.

**Design Principles:**

- Maintain consistent field names and data types.
- Represent missing information explicitly.
- Preserve evidence supporting extracted tasks.
- Separate action extraction from priority classification.
- Support reliable validation and downstream processing.

**Expected Outcome:**

Reusable Pydantic schemas that will serve as the foundation for our LangGraph multi-agent workflow.

In [11]:
# Step 4.1: Define structured output schemas

from typing import Literal
from pydantic import BaseModel, Field, ConfigDict

class TopicExtractionOutput(BaseModel):
    """Structured output for meeting topics."""

    model_config = ConfigDict(extra="forbid")

    topics: list[str] = Field(description="Main discussion topics identified in the meeting.")

class MeetingSummaryOutput(BaseModel):
    """Structured output for the meeting summary."""

    model_config = ConfigDict(extra="forbid")

    summary: str = Field(description="Concise meeting summary of 3 to 5 sentences.")

class ActionItem(BaseModel):
    """Represents an individual meeting action item."""

    model_config = ConfigDict(extra="forbid")

    task: str = Field(description="Clearly defined action or task.")
    owner: str = Field(description="Responsible speaker or 'Not specified'.")
    deadline: str | None = Field(description="Explicitly mentioned deadline, otherwise null.")
    source_evidence: str = Field(description="Supporting statement from the meeting transcript.")


class ActionExtractionOutput(BaseModel):
    """Structured output containing extracted action items."""

    model_config = ConfigDict(extra="forbid")

    action_items: list[ActionItem] = Field(description="List of identified meeting action items.")

class TaskPriority(BaseModel):
    """Represents the priority assigned to a task."""

    model_config = ConfigDict(extra="forbid")

    task: str = Field(description="Task description.")

    priority: Literal["High", "Medium", "Low"] = Field(description="Priority classification.")

class PriorityClassificationOutput(BaseModel):
    """Structured output for task priorities."""

    model_config = ConfigDict(extra="forbid")

    priorities: list[TaskPriority] = Field(description="Priority classifications for action items.")

# Display schema initialization results
print("=" * 65)
print("LANGGRAPH STRUCTURED OUTPUT SCHEMAS")
print("=" * 65)

schemas = [
    TopicExtractionOutput,
    MeetingSummaryOutput,
    ActionItem,
    ActionExtractionOutput,
    TaskPriority,
    PriorityClassificationOutput
]

for schema in schemas:
    print(f"{schema.__name__:<35} : Defined")

print("=" * 65)
print(f"Total Schemas Defined : {len(schemas)}")
print("Status: Structured output schemas initialized successfully.")

LANGGRAPH STRUCTURED OUTPUT SCHEMAS
TopicExtractionOutput               : Defined
MeetingSummaryOutput                : Defined
ActionItem                          : Defined
ActionExtractionOutput              : Defined
TaskPriority                        : Defined
PriorityClassificationOutput        : Defined
Total Schemas Defined : 6
Status: Structured output schemas initialized successfully.


## Step 4.2: Define LangGraph Shared State

**Objective:**

Create a centralized state structure that allows specialized AI agents to exchange information throughout the LangGraph workflow.

**What is Shared State?**

Shared state is a structured data object that stores the inputs, intermediate results, and final outputs of a LangGraph application.

Each workflow node reads the relevant state fields and returns updates to the shared state.

**State Components:**

1. `transcript` - Input meeting transcript.
2. `topics` - Key discussion topics.
3. `summary` - Generated meeting summary.
4. `action_items` - Extracted tasks, owners, deadlines, and evidence.
5. `priorities` - Priority classification for each task.
6. `final_report` - Consolidated meeting analysis.

**Implementation Approach:**

- Use Python TypedDict for state definition.
- Use existing Pydantic models for structured agent outputs.
- Define optional intermediate fields using NotRequired.
- Preserve the original transcript throughout the workflow.
- Keep state updates explicit and predictable.

**Expected Outcome:**

A reusable shared state schema ready for integration with LangGraph workflow nodes.

In [12]:
# Step 4.2: Define LangGraph shared state

from typing import TypedDict, NotRequired

class MeetingState(TypedDict):
    """Shared state for the AI Meeting Notes Analyzer."""

    # Required workflow input
    transcript: str

    # Intermediate agent outputs
    topics: NotRequired[list[str]]
    summary: NotRequired[str]
    action_items: NotRequired[list[ActionItem]]
    priorities: NotRequired[list[TaskPriority]]

    # Final workflow output
    final_report: NotRequired[str]


# Create initial workflow state
initial_state: MeetingState = {"transcript": cleaned_transcript}

# Validate required input
assert initial_state["transcript"].strip(), ("Meeting transcript cannot be empty.")

# Display shared state configuration
print("=" * 65)
print("LANGGRAPH SHARED STATE CONFIGURATION")
print("=" * 65)

print(f"State Schema          : {MeetingState.__name__}")
print(f"Total State Fields    : {len(MeetingState.__annotations__)}")
print(f"Required Fields       : {len(MeetingState.__required_keys__)}")
print(f"Optional Fields       : {len(MeetingState.__optional_keys__)}")

print("\nState Fields:")
print("-" * 65)

for field_name in MeetingState.__annotations__:
    requirement = (
        "Required"
        if field_name in MeetingState.__required_keys__
        else "Optional"
    )

    print(f"{field_name:<20} : {requirement}")

print("\nInitial State:")
print("-" * 65)

print(f"Transcript Loaded     : {bool(initial_state['transcript'])}")
print(f"Transcript Characters : {len(initial_state['transcript']):,}")

print("=" * 65)
print("Status: LangGraph shared state defined successfully.")

LANGGRAPH SHARED STATE CONFIGURATION
State Schema          : MeetingState
Total State Fields    : 6
Required Fields       : 1
Optional Fields       : 5

State Fields:
-----------------------------------------------------------------
transcript           : Required
topics               : Optional
summary              : Optional
action_items         : Optional
priorities           : Optional
final_report         : Optional

Initial State:
-----------------------------------------------------------------
Transcript Loaded     : True
Transcript Characters : 3,825
Status: LangGraph shared state defined successfully.


## Step 4.3: Develop the Topic Extraction Agent

**Objective:**

Develop the first specialized AI agent responsible for identifying the main discussion topics from the meeting transcript.

**Agent Responsibilities:**

1. Read the meeting transcript from LangGraph shared state.
2. Identify the major discussion topics.
3. Avoid extracting minor or unrelated conversational details.
4. Avoid introducing topics not supported by the transcript.
5. Return a structured list of topics.
6. Validate the output using the TopicExtractionOutput Pydantic schema.

**Implementation Approach:**

- Use the existing OpenAI gpt-4o-mini model.
- Use LangChain structured output functionality.
- Implement the agent as a reusable Python function.
- Accept MeetingState as input.
- Return only the topics field as a state update.

**Expected Output:**

A validated list of meeting discussion topics that can be passed to subsequent LangGraph workflow nodes.

**Evaluation Considerations:**

- Topic relevance
- Topic completeness
- Duplicate topics
- Unsupported or hallucinated topics
- Structured output validity

In [13]:
# Step 4.3: Develop and test Topic Extraction Agent

import os
from langchain_core.messages import SystemMessage, HumanMessage

# Apply previously identified SSL environment fix
os.environ.pop("SSL_CERT_FILE", None)

# Configure structured LLM output
topic_llm = llm.with_structured_output(
    TopicExtractionOutput,
    method="json_schema"
)

def topic_extraction_agent(state: MeetingState) -> dict:
    """Extract key discussion topics from a meeting transcript."""

    transcript = state["transcript"]

    if not transcript.strip():
        raise ValueError("Meeting transcript cannot be empty.")

    system_prompt = """
    You are an expert AI Meeting Topic Extraction Agent.

    Analyze the meeting transcript and identify the main discussion topics.

    Instructions:
    - Extract 3 to 8 meaningful discussion topics.
    - Use concise and descriptive topic names.
    - Focus on major issues, decisions, and discussion themes.
    - Avoid duplicate or overlapping topics.
    - Do not invent topics.
    - Preserve the meaning of the original discussion.
    - Return results using the required structured schema.
    """

    response = topic_llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=transcript)
    ])

    return {"topics": response.topics}

# Execute the agent independently
topic_result = topic_extraction_agent(initial_state)

# Display extracted topics
print("=" * 65)
print("TOPIC EXTRACTION AGENT - RESULTS")
print("=" * 65)

print(f"Total Topics Extracted : {len(topic_result['topics'])}")

print("\nKey Discussion Topics:")
print("-" * 65)

for index, topic in enumerate(topic_result["topics"], start=1):
    print(f"{index}. {topic}")

print("\n" + "=" * 65)
print("Status: Topic Extraction Agent executed successfully.")

TOPIC EXTRACTION AGENT - RESULTS
Total Topics Extracted : 8

Key Discussion Topics:
-----------------------------------------------------------------
1. Mobile App Crashing Issues
2. Root Cause Analysis of Login Failures
3. Impact of Recent Backend Changes
4. Testing on Different Android Versions
5. Communication Strategy for Affected Users
6. Error Handling Improvements in Mobile App
7. Documentation and Issue Tracking
8. Urgent Resolution and Patch Deployment

Status: Topic Extraction Agent executed successfully.


## Step 4.4: Develop the Meeting Summary Agent

**Objective:**

Develop a specialized AI agent that generates a concise and accurate meeting summary from the transcript.

**Agent Responsibilities:**

1. Read the meeting transcript from the shared state.
2. Identify the central meeting objective.
3. Summarize important discussion points and findings.
4. Capture decisions and agreed next steps.
5. Generate a summary containing 3–5 sentences.
6. Avoid unsupported assumptions and invented information.

**Implementation Approach:**

- Use the existing gpt-4o-mini model.
- Apply a specialized summarization system prompt.
- Use LangChain structured output functionality.
- Validate the response using MeetingSummaryOutput.
- Return the summary as a shared state update.

**Quality Considerations:**

- Factual accuracy
- Completeness
- Conciseness
- Clarity
- Hallucination prevention
- Preservation of uncertainty

**Expected Outcome:**

A structured meeting summary containing 3–5 meaningful sentences that accurately represent the meeting discussion.

In [14]:
# Step 4.4: Develop and test Meeting Summary Agent

import os
import re
from langchain_core.messages import SystemMessage, HumanMessage

# Apply previously identified SSL environment fix
os.environ.pop("SSL_CERT_FILE", None)

# Configure structured LLM output
summary_llm = llm.with_structured_output(
    MeetingSummaryOutput,
    method="json_schema"
)

def meeting_summary_agent(state: MeetingState) -> dict:
    """Generate a concise, factual meeting summary."""

    transcript = state["transcript"]

    if not transcript.strip():
        raise ValueError("Meeting transcript cannot be empty.")

    system_prompt = """
    You are an expert AI Meeting Summarization Agent.

    Generate a concise meeting summary using only the provided transcript.

    Instructions:
    - Write exactly 3 to 5 complete sentences.
    - Explain the main issue or meeting objective.
    - Include the most important findings.
    - Mention significant decisions and next steps.
    - Preserve uncertainty about suspected causes.
    - Do not invent facts, decisions, or outcomes.
    - Avoid unnecessary details and repetition.
    - Use clear, professional business language.
    - Return the result using the required schema.
    """

    response = summary_llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=transcript)
    ])

    return {"summary": response.summary}

# Execute the agent independently
summary_result = meeting_summary_agent(initial_state)

# Extract generated summary
generated_summary = summary_result["summary"]

# Estimate sentence count for initial validation
sentences = [
    sentence.strip()
    for sentence in re.split(r"(?<=[.!?])\s+", generated_summary)
    if sentence.strip()
]

sentence_count = len(sentences)

# Display results
print("=" * 65)
print("MEETING SUMMARY AGENT - RESULTS")
print("=" * 65)

print("\nGenerated Meeting Summary:")
print("-" * 65)
print(generated_summary)

print("\n" + "=" * 65)
print("SUMMARY VALIDATION")
print("=" * 65)

print(f"Sentence Count : {sentence_count}")
print(f"Word Count     : {len(generated_summary.split())}")

if 3 <= sentence_count <= 5:
    print("Length Check   : Passed")
else:
    print("Length Check   : Review Required")

print("=" * 65)
print("Status: Meeting Summary Agent executed successfully.")

MEETING SUMMARY AGENT - RESULTS

Generated Meeting Summary:
-----------------------------------------------------------------
The meeting addressed urgent complaints regarding the mobile app crashing during login, particularly affecting Android users. The team identified that the issue may be linked to a recent update to the authentication API deployed three days ago, coinciding with the spike in crash reports. Action items include the Mobile Developer reviewing the login module, the QA Tester preparing a detailed bug report, and the Backend Developer verifying the API response format. The team agreed to prioritize these investigations and communicate updates to both management and affected users. A follow-up meeting is scheduled for tomorrow morning to assess progress.

SUMMARY VALIDATION
Sentence Count : 5
Word Count     : 98
Length Check   : Passed
Status: Meeting Summary Agent executed successfully.


## Step 4.5: Develop the Action Item Extraction Agent

**Objective:**

Develop a specialized AI agent that extracts actionable tasks, responsible owners, deadlines, and supporting evidence from the meeting transcript.

**Agent Responsibilities:**

1. Identify explicit task commitments and assignments.
2. Identify clearly requested or agreed follow-up actions.
3. Extract concise and actionable task descriptions.
4. Associate tasks with the correct meeting participants.
5. Extract deadlines when explicitly mentioned.
6. Preserve supporting transcript evidence.
7. Avoid duplicate or overlapping action items.
8. Avoid inventing task owners or deadlines.

**Ownership Rules:**

- Assign an owner when explicitly identified.
- Resolve first-person commitments using the speaker label.
- Use "Not specified" when ownership is unclear.
- Do not assume that a participant owns a task merely because they discussed it.

**Deadline Rules:**

- Preserve relative deadlines as stated in the transcript.
- Do not invent calendar dates.
- Use null when no deadline is specified.

**Structured Output:**

ActionExtractionOutput:
- action_items
  - task
  - owner
  - deadline
  - source_evidence

**Evaluation Criteria:**

- Action item completeness
- Task ownership accuracy
- Deadline extraction accuracy
- Evidence consistency
- Duplicate detection
- Unsupported information

**Expected Outcome:**

A validated collection of action items ready for priority classification and integration into the LangGraph workflow.

In [15]:
# Step 4.5: Develop and test Action Item Extraction Agent

import os
from langchain_core.messages import SystemMessage, HumanMessage

# Apply previously identified SSL environment fix
os.environ.pop("SSL_CERT_FILE", None)

# Configure structured LLM output
action_llm = llm.with_structured_output(
    ActionExtractionOutput,
    method="json_schema"
)

def action_item_extraction_agent(state: MeetingState) -> dict:
    """Extract meeting action items with owners, deadlines, and evidence."""

    transcript = state["transcript"]

    if not transcript.strip():
        raise ValueError("Meeting transcript cannot be empty.")

    system_prompt = """
    You are an expert AI Meeting Action Item Extraction Agent.

    Extract all meaningful action items from the meeting transcript.

    EXTRACTION RULES:

    1. Identify explicit commitments, assigned tasks, and clearly agreed follow-up actions.

    2. Convert each action into a concise task description.

    3. Identify the responsible owner:
       - For "I will" statements, use the current speaker.
       - For explicitly assigned tasks, use the named owner.
       - If no owner is identified, use "Not specified".

    4. Extract deadlines exactly as expressed.
       Do not convert relative deadlines into calendar dates.
       Use null when no deadline is provided.

    5. Include the supporting transcript statement in source_evidence.

    6. Do not invent owners, deadlines, or commitments.

    7. Do not treat questions, observations, or speculation as confirmed assignments.

    8. Include clearly agreed actions even if the owner is not specified.

    9. Merge duplicate references to the same task, but preserve distinct deliverables.

    10. Return an empty action_items list if no actionable tasks are identified.

    Return results using the required structured schema.
    """

    response = action_llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=transcript)
    ])

    return {"action_items": response.action_items}

# Execute agent independently
action_result = action_item_extraction_agent(initial_state)

# Extract action items
extracted_actions = action_result["action_items"]

# Display results
print("=" * 80)
print("ACTION ITEM EXTRACTION AGENT - RESULTS")
print("=" * 80)

print(f"\nTotal Action Items Extracted: {len(extracted_actions)}")

for index, action in enumerate(extracted_actions, start=1):

    print("\n" + "-" * 80)
    print(f"ACTION ITEM {index}")
    print("-" * 80)

    print(f"Task     : {action.task}")
    print(f"Owner    : {action.owner}")
    print(f"Deadline : {action.deadline or 'Not specified'}")
    print(f"Evidence : {action.source_evidence}")

print("\n" + "=" * 80)
print("Status: Action Item Extraction Agent executed successfully.")

ACTION ITEM EXTRACTION AGENT - RESULTS

Total Action Items Extracted: 20

--------------------------------------------------------------------------------
ACTION ITEM 1
--------------------------------------------------------------------------------
Task     : Review the Android login module and check if any error handling is missing.
Owner    : Mobile Developer
Deadline : null
Evidence : Mobile Developer: I will review the Android login module and check if any error handling is missing.

--------------------------------------------------------------------------------
ACTION ITEM 2
--------------------------------------------------------------------------------
Task     : Prepare a detailed bug report with screenshots and logs.
Owner    : QA Tester
Deadline : null
Evidence : QA Tester: I will prepare a detailed bug report with screenshots and logs.

--------------------------------------------------------------------------------
ACTION ITEM 3
-------------------------------------------

## Step 4.5.1: Validate Extracted Action Items

**Objective:**

Evaluate the quality and consistency of action items generated by the Action Item Extraction Agent.

**Validation Checks:**

1. Verify that supporting evidence exists in the transcript.
2. Identify action items containing deadline expressions.
3. Detect missing deadlines when timing information is present.
4. Validate task owners against known meeting participants.
5. Flag potential ownership ambiguities for manual review.

**Methodology:**

Use deterministic Python validation rules without additional LLM calls.

**Important Considerations:**

- Evidence matching does not guarantee semantic correctness.
- Deadline detection is rule-based and may miss unusual expressions.
- Ownership ambiguity checks are diagnostic, not automatic corrections.
- Original extracted results will remain unchanged.

**Expected Outcome:**

A validation report identifying potential issues that should be addressed before priority classification and LangGraph integration.

In [16]:
# Step 4.5.1: Validate extracted action items

import re

# Known participants
valid_owners = set(speaker_counts.keys()) | {"Not specified"}

# Deadline expressions from the meeting context
deadline_patterns = [
    r"\bby tomorrow\b",
    r"\btomorrow morning\b",
    r"\bwithin the next hour\b",
    r"\bby the end of the day\b",
    r"\bend of the day\b",
    r"\bright away\b",
    r"\bimmediately\b",
    r"\btoday\b",
    r"\bthis week\b"
]

validation_results = []

for index, action in enumerate(extracted_actions, start=1):

    # Validate supporting evidence
    evidence_valid = action.source_evidence.strip() in cleaned_transcript

    # Validate owner
    owner_valid = action.owner in valid_owners

    # Detect timing expressions
    combined_text = f"{action.task} {action.source_evidence}"

    detected_deadlines = []

    for pattern in deadline_patterns:
        matches = re.findall(pattern, combined_text, flags=re.IGNORECASE)
        detected_deadlines.extend(matches)

    # Remove duplicate deadline expressions
    detected_deadlines = list(
        dict.fromkeys(
            deadline.lower() for deadline in detected_deadlines
        )
    )

    # Identify missing deadlines
    deadline_missing = (
        bool(detected_deadlines)
        and not action.deadline
    )

    # Flag proposals with potentially inferred ownership
    proposal_pattern = r"\b(?:we should|let's|we need to)\b"

    ownership_review = (
        action.owner != "Not specified"
        and bool(
            re.search(proposal_pattern, action.source_evidence, flags=re.IGNORECASE)
        )
    )

    validation_results.append({
        "action_number": index,
        "evidence_valid": evidence_valid,
        "owner_valid": owner_valid,
        "detected_deadlines": detected_deadlines,
        "deadline_missing": deadline_missing,
        "ownership_review": ownership_review
    })

# Summarize validation results
total_actions = len(validation_results)

valid_evidence_count = sum(
    result["evidence_valid"] for result in validation_results
)

valid_owner_count = sum(
    result["owner_valid"] for result in validation_results
)

missing_deadline_count = sum(
    result["deadline_missing"] for result in validation_results
)

ownership_review_count = sum(
    result["ownership_review"] for result in validation_results
)

print("=" * 75)
print("ACTION ITEM VALIDATION REPORT")
print("=" * 75)

print(f"Total Action Items        : {total_actions}")
print(f"Valid Evidence Statements : {valid_evidence_count}")
print(f"Valid Owner Labels        : {valid_owner_count}")
print(f"Missing Deadline Flags    : {missing_deadline_count}")
print(f"Ownership Review Flags    : {ownership_review_count}")

print("\n" + "-" * 75)
print("ITEMS REQUIRING REVIEW")
print("-" * 75)

for result in validation_results:

    issues = []

    if not result["evidence_valid"]:
        issues.append("Evidence not found")

    if not result["owner_valid"]:
        issues.append("Invalid owner")

    if result["deadline_missing"]:
        issues.append(
            "Missing deadline: "
            + ", ".join(result["detected_deadlines"])
        )

    if result["ownership_review"]:
        issues.append("Potential ownership ambiguity")

    if issues:
        print(
            f"Action {result['action_number']:>2}: "
            + " | ".join(issues)
        )

print("\n" + "=" * 75)
print("Status: Action item validation completed.")

ACTION ITEM VALIDATION REPORT
Total Action Items        : 20
Valid Evidence Statements : 20
Valid Owner Labels        : 20
Missing Deadline Flags    : 0
Ownership Review Flags    : 1

---------------------------------------------------------------------------
ITEMS REQUIRING REVIEW
---------------------------------------------------------------------------
Action 14: Potential ownership ambiguity

Status: Action item validation completed.


## Step 4.5.2: Improve Deadline and Ownership Validation

**Objective:**

Improve the validation logic for missing deadlines and potentially incorrect task ownership.

**Issues Identified:**

The previous validation reported zero missing deadlines even though several extracted action items contain explicit timing expressions.

This may occur when missing deadlines are represented as text values such as "null" instead of Python None.

**Validation Improvements:**

1. Normalize missing deadline representations.
2. Detect timing expressions in source evidence.
3. Identify actions where deadlines are missing.
4. Detect potential ownership assumptions.
5. Preserve original extracted results for comparison.

**Expected Outcome:**

An improved validation report highlighting missing deadlines and ownership ambiguities requiring correction.

In [17]:
# Step 4.5.2: Improved deadline and ownership validation

import re

# Missing deadline representations
missing_values = {
    "",
    "null",
    "none",
    "not specified",
    "n/a",
    "unknown"
}

# Timing expressions
timing_patterns = [
    r"\bby tomorrow\b",
    r"\btomorrow morning\b",
    r"\bwithin the next hour\b",
    r"\bby the end of the day\b",
    r"\bend of the day\b",
    r"\bright away\b",
    r"\bimmediately\b",
    r"\btoday\b",
    r"\bthis week\b",
    r"\bafter this meeting\b"
]

improved_validation_results = []

for index, action in enumerate(extracted_actions, start=1):

    # Normalize deadline representation
    raw_deadline = action.deadline

    normalized_deadline = (
        ""
        if raw_deadline is None
        else str(raw_deadline).strip().lower()
    )

    deadline_missing = normalized_deadline in missing_values

    # Detect timing expressions in source evidence
    evidence_text = action.source_evidence

    detected_timing = []

    for pattern in timing_patterns:
        matches = re.findall(
            pattern,
            evidence_text,
            flags=re.IGNORECASE
        )
        detected_timing.extend(matches)

    detected_timing = list(dict.fromkeys(detected_timing))

    missing_explicit_deadline = (
        deadline_missing and bool(detected_timing)
    )

    # Detect potential inferred ownership
    proposal_pattern = r"\b(?:we should|let's|we need to)\b"

    ownership_review = (
        action.owner != "Not specified"
        and bool(
            re.search(
                proposal_pattern,
                evidence_text,
                flags=re.IGNORECASE
            )
        )
    )

    improved_validation_results.append({
        "action_number": index,
        "task": action.task,
        "owner": action.owner,
        "raw_deadline": raw_deadline,
        "detected_timing": detected_timing,
        "missing_deadline": missing_explicit_deadline,
        "ownership_review": ownership_review
    })

# Summary
missing_deadlines = [
    item for item in improved_validation_results
    if item["missing_deadline"]
]

ownership_flags = [
    item for item in improved_validation_results
    if item["ownership_review"]
]

print("=" * 75)
print("IMPROVED ACTION ITEM VALIDATION REPORT")
print("=" * 75)

print(f"Total Action Items       : {len(extracted_actions)}")
print(f"Missing Deadline Flags   : {len(missing_deadlines)}")
print(f"Ownership Review Flags   : {len(ownership_flags)}")

print("\n" + "-" * 75)
print("MISSING DEADLINES")
print("-" * 75)

for item in missing_deadlines:
    print(
        f"Action {item['action_number']:>2}: "
        f"{', '.join(item['detected_timing'])}"
    )

print("\n" + "-" * 75)
print("OWNERSHIP AMBIGUITIES")
print("-" * 75)

for item in ownership_flags:
    print(
        f"Action {item['action_number']:>2}: "
        f"{item['task']} | Owner: {item['owner']}"
    )

print("\n" + "=" * 75)
print("Status: Improved validation completed.")

IMPROVED ACTION ITEM VALIDATION REPORT
Total Action Items       : 20
Missing Deadline Flags   : 6
Ownership Review Flags   : 1

---------------------------------------------------------------------------
MISSING DEADLINES
---------------------------------------------------------------------------
Action  7: by tomorrow
Action 15: immediately, after this meeting
Action 16: right away
Action 17: within the next hour
Action 18: by the end of the day, end of the day
Action 20: tomorrow morning

---------------------------------------------------------------------------
OWNERSHIP AMBIGUITIES
---------------------------------------------------------------------------
Action 14: Release the patch update to the Play Store after testing confirms the issue is resolved. | Owner: QA Tester

Status: Improved validation completed.


## Step 4.5.3: Refine the Action Item Extraction Agent

**Objective:**

Improve action item extraction accuracy based on the validation findings.

**Identified Issues:**

- Six action items contain timing expressions but have missing deadlines.
- One action item has a potentially incorrect owner.
- Some extracted actions overlap.

**Refinement Strategy:**

1. Explicitly instruct the model to populate the deadline field.
2. Preserve relative deadlines without inventing calendar dates.
3. Distinguish commitments from proposals.
4. Use "Not specified" when ownership is not established.
5. Consolidate duplicate tasks without losing distinct deliverables.
6. Preserve exact supporting transcript evidence.
7. Maintain structured Pydantic output.

**Evaluation:**

Compare the refined results against the original 20 extracted actions.

Review:
- Total action items
- Deadlines captured
- Owner assignments
- Evidence validity
- Missing or duplicated tasks

**Expected Outcome:**

A more accurate and consistent collection of meeting action items suitable for priority classification.

In [18]:
# Step 4.5.3: Refine the Action Item Extraction Agent

import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Apply previously identified SSL environment fix
os.environ.pop("SSL_CERT_FILE", None)

# Increase output allowance for detailed structured extraction
refined_action_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=4000,
    timeout=60,
    max_retries=2
).with_structured_output(
    ActionExtractionOutput,
    method="json_schema"
)


def refined_action_extraction_agent(state: MeetingState) -> dict:
    """Extract action items with improved ownership and deadline accuracy."""

    transcript = state["transcript"]

    if not transcript.strip():
        raise ValueError("Meeting transcript cannot be empty.")

    system_prompt = """
    You are a highly accurate AI Meeting Action Item Extraction Agent.

    Extract actionable tasks from the transcript using the rules below.

    TASK EXTRACTION:
    - Identify explicit commitments, assignments, and agreed actions.
    - Include conditional commitments and preserve their conditions.
    - Do not turn ordinary observations into tasks.
    - Combine duplicate references to the same deliverable.
    - Preserve distinct tasks when they have different outcomes.

    OWNER EXTRACTION:
    - For "I will" commitments, assign the current speaker.
    - For explicitly assigned tasks, use the identified owner.
    - For suggestions such as "we should" or "let's",
      do not automatically assign the speaker as the owner.
    - Use "Not specified" when no responsible owner is established.
    - Do not infer ownership from a participant's job title.

    DEADLINE EXTRACTION:
    - Examine every action for timing information.
    - Populate the deadline field when timing is explicitly stated.
    - Preserve expressions such as:
      "by tomorrow", "tomorrow morning",
      "within the next hour", "by the end of the day",
      "right away", and "immediately after this meeting".
    - For event-dependent timing such as "once the fix is ready",
      preserve the condition when it functions as a task trigger.
    - Do not invent dates or times.
    - Use JSON null only when no deadline or timing is specified.
    - Never use the text strings "null", "None", or "N/A"
      to represent a missing deadline.

    EVIDENCE:
    - Copy the supporting statement exactly from the transcript.
    - Include the speaker label when present.
    - Never fabricate supporting evidence.

    QUALITY:
    - Avoid duplicate tasks.
    - Preserve meaningful distinct commitments.
    - Ensure every owner is supported by evidence.
    - Ensure every stated deadline is captured.
    - Return results using the required Pydantic schema.
    """

    response = refined_action_llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=transcript)
    ])

    return {"action_items": response.action_items}


# Run refined extraction
refined_action_result = refined_action_extraction_agent(initial_state)

refined_actions = refined_action_result["action_items"]

# Count populated deadlines
missing_values = {
    "", "null", "none", "not specified", "n/a", "unknown"
}

deadlines_captured = sum(
    action.deadline is not None
    and str(action.deadline).strip().lower() not in missing_values
    for action in refined_actions
)

# Count unassigned owners
unassigned_owners = sum(
    action.owner == "Not specified"
    for action in refined_actions
)

print("=" * 80)
print("REFINED ACTION ITEM EXTRACTION - RESULTS")
print("=" * 80)

print(f"Original Action Items : {len(extracted_actions)}")
print(f"Refined Action Items  : {len(refined_actions)}")
print(f"Deadlines Captured    : {deadlines_captured}")
print(f"Unassigned Owners     : {unassigned_owners}")

print("\n" + "=" * 80)
print("REFINED ACTION ITEMS")
print("=" * 80)

for index, action in enumerate(refined_actions, start=1):

    print(f"\nACTION ITEM {index}")
    print("-" * 80)

    print(f"Task     : {action.task}")
    print(f"Owner    : {action.owner}")
    print(f"Deadline : {action.deadline}")
    print(f"Evidence : {action.source_evidence}")

print("\n" + "=" * 80)
print("Status: Refined extraction executed successfully.")

REFINED ACTION ITEM EXTRACTION - RESULTS
Original Action Items : 20
Refined Action Items  : 15
Deadlines Captured    : 5
Unassigned Owners     : 4

REFINED ACTION ITEMS

ACTION ITEM 1
--------------------------------------------------------------------------------
Task     : Review the Android login module and check if any error handling is missing.
Owner    : Mobile Developer
Deadline : by tomorrow.
Evidence : Mobile Developer: I will review the Android login module and check if any error handling is missing.

ACTION ITEM 2
--------------------------------------------------------------------------------
Task     : Prepare a detailed bug report with screenshots and logs.
Owner    : QA Tester
Deadline : None
Evidence : QA Tester: I will prepare a detailed bug report with screenshots and logs.

ACTION ITEM 3
--------------------------------------------------------------------------------
Task     : Verify the authentication API and check if there are any breaking changes.
Owner    : Back

## Step 4.5.4: Compare Original and Refined Action Items

**Objective:**

Compare the original and refined action extraction results to identify potential information loss.

**Comparison Methodology:**

1. Compare each original task against all refined tasks.
2. Identify the closest textual match.
3. Calculate a similarity score.
4. Flag tasks with low similarity for manual review.
5. Examine whether consolidation preserved important commitments.

**Evaluation Considerations:**

- Task completeness
- Task consolidation
- Missing commitments
- Deadline preservation
- Ownership accuracy

**Important Limitation:**

Text similarity does not establish semantic equivalence. Flagged items require review against the original meeting transcript.

**Expected Outcome:**

A comparison report identifying original action items that may have been omitted or substantially changed during refinement.

In [19]:
# Step 4.5.4: Compare original and refined action items

from difflib import SequenceMatcher

def calculate_similarity(text1, text2):
    """Calculate normalized textual similarity."""

    return SequenceMatcher(
        None,
        text1.lower().strip(),
        text2.lower().strip()
    ).ratio()


comparison_results = []

for original_index, original_action in enumerate(extracted_actions, start=1):

    matches = []

    for refined_index, refined_action in enumerate(refined_actions, start=1):

        similarity = calculate_similarity(
            original_action.task,
            refined_action.task
        )

        matches.append({
            "refined_index": refined_index,
            "similarity": similarity,
            "refined_task": refined_action.task
        })

    best_match = max(
        matches,
        key=lambda item: item["similarity"]
    )

    comparison_results.append({
        "original_index": original_index,
        "original_task": original_action.task,
        "best_match": best_match
    })


# Threshold for identifying tasks requiring review
similarity_threshold = 0.65

review_items = [
    item for item in comparison_results
    if item["best_match"]["similarity"] < similarity_threshold
]

print("=" * 80)
print("ORIGINAL VS REFINED ACTION ITEM COMPARISON")
print("=" * 80)

print(f"Original Action Items      : {len(extracted_actions)}")
print(f"Refined Action Items       : {len(refined_actions)}")
print(f"Similarity Threshold       : {similarity_threshold:.0%}")
print(f"Items Flagged for Review   : {len(review_items)}")

print("\n" + "=" * 80)
print("POTENTIALLY MISSING OR MODIFIED ACTIONS")
print("=" * 80)

for item in review_items:

    match = item["best_match"]

    print(f"\nOriginal Action {item['original_index']}")
    print("-" * 80)

    print(f"Original Task : {item['original_task']}")
    print(f"Closest Match : {match['refined_task']}")
    print(f"Refined Item  : {match['refined_index']}")
    print(f"Similarity    : {match['similarity']:.2%}")

print("\n" + "=" * 80)
print("Status: Action item comparison completed.")

ORIGINAL VS REFINED ACTION ITEM COMPARISON
Original Action Items      : 20
Refined Action Items       : 15
Similarity Threshold       : 65%
Items Flagged for Review   : 5

POTENTIALLY MISSING OR MODIFIED ACTIONS

Original Action 7
--------------------------------------------------------------------------------
Original Task : Push a fix for the login module by tomorrow if the issue is found there.
Closest Match : Provide an update on the progress by the end of the day.
Refined Item  : 11
Similarity    : 43.75%

Original Action 8
--------------------------------------------------------------------------------
Original Task : Perform regression testing once the fix is ready.
Closest Match : Prepare a communication message for users once the fix is deployed.
Refined Item  : 8
Similarity    : 55.17%

Original Action 9
--------------------------------------------------------------------------------
Original Task : Test the authentication endpoints after the fix is deployed.
Closest Match : 

## Step 4.5.5: Reconcile and Finalize Action Items

**Objective:**

Create a consolidated and validated action item collection using the original extraction and the findings from subsequent evaluation.

**Reconciliation Strategy:**

1. Preserve all meaningful original commitments.
2. Combine preparation and sharing of the bug report.
3. Combine server-log and immediate API-log reviews.
4. Combine immediate fix work with the conditional fix deadline.
5. Correct unsupported ownership assignments.
6. Preserve explicit relative deadlines.
7. Retain multiple evidence statements for consolidated tasks.

**Validation:**

- Confirm all supporting evidence appears in the transcript.
- Confirm every owner is a recognized participant.
- Confirm the corrected deadlines are preserved.
- Confirm no original action is accidentally discarded.

**Expected Outcome:**

A reconciled action item collection ready for the Priority Classification Agent.

**Note:**

These are evidence-based corrections for the current sample transcript. A future production workflow must apply equivalent checks automatically to new transcripts.

In [20]:
# Step 4.5.5: Reconcile and validate action items

# Start from the original extraction
# Original action numbers are 1-based.

actions_by_id = {
    index: action.model_copy(deep=True)
    for index, action in enumerate(extracted_actions, start=1)
}

# Helper to combine exact evidence statements
def combine_evidence(*action_ids):
    return "\n".join(
        actions_by_id[action_id].source_evidence
        for action_id in action_ids
    )


# 1. Prepare and share bug report within one hour
actions_by_id[2] = actions_by_id[2].model_copy(
    update={
        "task": (
            "Prepare and share the detailed bug report "
            "with screenshots and logs."
        ),
        "deadline": "Within the next hour",
        "source_evidence": combine_evidence(2, 17)
    }
)

# 2. Consolidate server/API log investigation
actions_by_id[5] = actions_by_id[5].model_copy(
    update={
        "task": (
            "Review server and API logs for "
            "authentication errors."
        ),
        "deadline": "Right away",
        "source_evidence": combine_evidence(5, 16)
    }
)

# 3. Consolidate immediate fix work and conditional delivery
actions_by_id[7] = actions_by_id[7].model_copy(
    update={
        "task": (
            "Begin working on the login fix immediately; "
            "if the issue is in the login module, "
            "aim to push the fix by tomorrow."
        ),
        "deadline": (
            "Start immediately after the meeting; "
            "conditional fix target: by tomorrow"
        ),
        "source_evidence": combine_evidence(7, 15)
    }
)

# 4. Correct Play Store release ownership
actions_by_id[14] = actions_by_id[14].model_copy(
    update={
        "owner": "Not specified",
        "deadline": "After successful testing and confirmation"
    }
)

# 5. Preserve explicit progress-update deadline
actions_by_id[18] = actions_by_id[18].model_copy(
    update={"deadline": "By the end of the day"}
)

# 6. Preserve follow-up meeting timing
actions_by_id[20] = actions_by_id[20].model_copy(
    update={"deadline": "Tomorrow morning"}
)

# Remove actions already consolidated into other items
merged_action_ids = {15, 16, 17}

final_actions = [
    action
    for action_id, action in actions_by_id.items()
    if action_id not in merged_action_ids
]

# Validate evidence line by line
evidence_checks = [
    all(
        evidence_line.strip() in cleaned_transcript
        for evidence_line in action.source_evidence.splitlines()
        if evidence_line.strip()
    )
    for action in final_actions
]

owner_checks = [
    action.owner in valid_owners
    for action in final_actions
]

assert all(evidence_checks), "Some evidence is missing from transcript."
assert all(owner_checks), "An invalid owner was detected."

# Save reconciled actions to the shared state
validated_state = {
    **initial_state,
    "topics": topic_result["topics"],
    "summary": summary_result["summary"],
    "action_items": final_actions
}

print("=" * 80)
print("FINAL ACTION ITEM RECONCILIATION REPORT")
print("=" * 80)

print(f"Original Action Items   : {len(extracted_actions)}")
print(f"Consolidated Items      : {len(final_actions)}")
print(f"Evidence Checks Passed  : {sum(evidence_checks)}/{len(final_actions)}")
print(f"Owner Checks Passed     : {sum(owner_checks)}/{len(final_actions)}")

print("\n" + "=" * 80)
print("FINAL CONSOLIDATED ACTION ITEMS")
print("=" * 80)

for index, action in enumerate(final_actions, start=1):
    print(f"\nAction {index}: {action.task}")
    print(f"Owner    : {action.owner}")
    print(f"Deadline : {action.deadline or 'Not specified'}")

print("\n" + "=" * 80)
print("Status: Reconciliation and validation completed.")

FINAL ACTION ITEM RECONCILIATION REPORT
Original Action Items   : 20
Consolidated Items      : 17
Evidence Checks Passed  : 17/17
Owner Checks Passed     : 17/17

FINAL CONSOLIDATED ACTION ITEMS

Action 1: Review the Android login module and check if any error handling is missing.
Owner    : Mobile Developer
Deadline : null

Action 2: Prepare and share the detailed bug report with screenshots and logs.
Owner    : QA Tester
Deadline : Within the next hour

Action 3: Verify the authentication API and check if there are any breaking changes.
Owner    : Backend Developer
Deadline : null

Action 4: Test the login flow on Android 11, 12, and 13.
Owner    : QA Tester
Deadline : null

Action 5: Review server and API logs for authentication errors.
Owner    : Backend Developer
Deadline : Right away

Action 6: Send a message to the support team to update customers about the fix.
Owner    : Customer Support Lead
Deadline : null

Action 7: Begin working on the login fix immediately; if the issue i

## Step 4.6: Develop the Priority Classification Agent

**Objective:**

Develop a specialized AI agent that classifies meeting action items into High, Medium, or Low priority.

**Agent Responsibilities:**

1. Read the meeting transcript and extracted action items.
2. Evaluate urgency, deadlines, and customer impact.
3. Consider task dependencies and operational importance.
4. Assign exactly one priority to each action item.
5. Preserve original task descriptions.
6. Avoid adding or removing action items.
7. Return structured priority classifications.

**Priority Classification Guidelines:**

**High Priority:**
- Urgent production issues.
- Critical customer-impacting tasks.
- Immediate or near-term deadlines.
- Tasks blocking incident resolution.

**Medium Priority:**
- Important testing and follow-up activities.
- Supporting investigation or coordination tasks.
- Tasks with moderate urgency.

**Low Priority:**
- Routine administrative work.
- Nonurgent documentation.
- Improvements without immediate operational impact.

**Implementation Approach:**

- Use the existing gpt-4o-mini model.
- Apply a specialized priority classification prompt.
- Validate output using PriorityClassificationOutput.
- Return classifications as a LangGraph state update.

**Evaluation Criteria:**

- Priority consistency
- Alignment with meeting context
- Urgency and deadline awareness
- Completeness
- Task preservation
- Structured output validity

**Expected Outcome:**

A validated priority classification for every action item.

In [21]:
# Step 4.6: Develop and test Priority Classification Agent

import os
import json
from langchain_core.messages import SystemMessage, HumanMessage

# Apply previously identified SSL environment fix
os.environ.pop("SSL_CERT_FILE", None)

# Configure structured LLM output
priority_llm = llm.with_structured_output(
    PriorityClassificationOutput,
    method="json_schema"
)


def priority_classification_agent(state: MeetingState) -> dict:
    """Classify meeting action items as High, Medium, or Low."""

    transcript = state["transcript"]
    action_items = state.get("action_items", [])

    # Support conditional routing when no actions exist
    if not action_items:
        return {"priorities": []}

    # Prepare action items with task identifiers
    action_payload = [
        {
            "id": index,
            "task": action.task,
            "owner": action.owner,
            "deadline": (
                None
                if action.deadline is None
                or str(action.deadline).strip().lower()
                in {"null", "none", "", "not specified"}
                else action.deadline
            )
        }
        for index, action in enumerate(action_items, start=1)
    ]

    system_prompt = """
    You are an expert AI Meeting Task Priority Classification Agent.

    Classify every provided action item as High, Medium, or Low.

    PRIORITY RULES:

    HIGH:
    - Urgent production incidents.
    - Tasks directly affecting customers.
    - Immediate or near-term deadlines.
    - Work blocking critical issue resolution.

    MEDIUM:
    - Important testing, coordination, and follow-up.
    - Supporting tasks with moderate urgency.
    - Work required before completing other tasks.

    LOW:
    - Routine or nonurgent administrative work.
    - Tasks without immediate operational impact.

    IMPORTANT INSTRUCTIONS:

    1. Consider the full meeting context.
    2. Classify every action item exactly once.
    3. Preserve each original task description exactly.
    4. Do not invent new tasks.
    5. Do not omit or duplicate tasks.
    6. Use only High, Medium, or Low.
    7. Return the required structured schema.
    """

    response = priority_llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(
            content=(
                "MEETING TRANSCRIPT:\n"
                + transcript
                + "\n\nACTION ITEMS:\n"
                + json.dumps(action_payload, indent=2)
            )
        )
    ])

    # Validate task coverage using task text
    original_tasks = [action.task for action in action_items]
    classified_tasks = [item.task for item in response.priorities]

    if (
        len(classified_tasks) != len(original_tasks)
        or sorted(classified_tasks) != sorted(original_tasks)
    ):
        raise ValueError(
            "Priority classification did not preserve "
            "all original action items exactly."
        )

    return {"priorities": response.priorities}


# Execute agent using reconciled action items
priority_result = priority_classification_agent(validated_state)

classified_priorities = priority_result["priorities"]

# Calculate distribution
priority_counts = {
    level: sum(
        item.priority == level
        for item in classified_priorities
    )
    for level in ["High", "Medium", "Low"]
}

print("=" * 80)
print("PRIORITY CLASSIFICATION AGENT - RESULTS")
print("=" * 80)

print(f"\nTotal Action Items : {len(final_actions)}")
print(f"Classified Tasks   : {len(classified_priorities)}")

print("\nPRIORITY DISTRIBUTION")
print("-" * 80)

for level, count in priority_counts.items():
    print(f"{level:<8}: {count}")

print("\n" + "=" * 80)
print("DETAILED PRIORITY CLASSIFICATION")
print("=" * 80)

for index, item in enumerate(classified_priorities, start=1):
    print(f"\nAction {index}")
    print(f"Priority : {item.priority}")
    print(f"Task     : {item.task}")

print("\n" + "=" * 80)
print("Status: Priority Classification Agent executed successfully.")

PRIORITY CLASSIFICATION AGENT - RESULTS

Total Action Items : 17
Classified Tasks   : 17

PRIORITY DISTRIBUTION
--------------------------------------------------------------------------------
High    : 5
Medium  : 9
Low     : 3

DETAILED PRIORITY CLASSIFICATION

Action 1
Priority : High
Task     : Review the Android login module and check if any error handling is missing.

Action 2
Priority : High
Task     : Prepare and share the detailed bug report with screenshots and logs.

Action 3
Priority : High
Task     : Verify the authentication API and check if there are any breaking changes.

Action 4
Priority : Medium
Task     : Test the login flow on Android 11, 12, and 13.

Action 5
Priority : High
Task     : Review server and API logs for authentication errors.

Action 6
Priority : Medium
Task     : Send a message to the support team to update customers about the fix.

Action 7
Priority : High
Task     : Begin working on the login fix immediately; if the issue is in the login module, ai

## Step 4.7: Build the LangGraph Workflow Structure

**Objective:**

Integrate the specialized AI agents into a LangGraph workflow with conditional routing.

**Workflow Sequence:**

1. START
2. Topic Extraction Agent
3. Meeting Summary Agent
4. Action Item Extraction Agent
5. Conditional Router
6. Priority Classification Agent (when actions exist)
7. Final Report Generator
8. END

**Conditional Routing Logic:**

- If action items exist, execute the Priority Classification Agent.
- If no action items exist, skip priority classification.
- Both routes continue to the Final Report Generator.

**Implementation Approach:**

- Use LangGraph StateGraph.
- Reuse the existing MeetingState schema.
- Register the specialized agents as graph nodes.
- Define sequential transitions.
- Implement conditional routing.
- Compile the workflow after all nodes are connected.

**Expected Outcome:**

A compiled LangGraph workflow ready for end-to-end testing.

**Note:**

The workflow will initially use the reusable extraction agents. The manually reconciled action list remains a reference for evaluating their output, not a hardcoded production result.

In [23]:
# Step 4.7: Build the LangGraph workflow structure

from langgraph.graph import StateGraph, START, END

# Initialize workflow using our shared state schema
workflow = StateGraph(MeetingState)


# Conditional routing function
def route_after_action_extraction(state: MeetingState) -> str:
    """Route based on whether action items were extracted."""

    action_items = state.get("action_items", [])

    if action_items:
        return "priority_classification"

    return "final_report"


# Temporary final report node
# We'll replace this with a formatted report generator next.
def final_report_node(state: MeetingState) -> dict:
    """Create a temporary report placeholder."""

    return {
        "final_report": (
            "Meeting analysis completed. "
            "Detailed report formatting will be added next."
        )
    }


# Register specialized agent nodes
workflow.add_node("topic_extraction", topic_extraction_agent)

workflow.add_node("meeting_summary", meeting_summary_agent)

workflow.add_node("action_extraction", refined_action_extraction_agent)

workflow.add_node("priority_classification", priority_classification_agent)

workflow.add_node("final_report", final_report_node)


# Define workflow transitions
workflow.add_edge(START, "topic_extraction")

workflow.add_edge("topic_extraction", "meeting_summary")

workflow.add_edge("meeting_summary", "action_extraction")

# Conditional routing
workflow.add_conditional_edges(
    "action_extraction",
    route_after_action_extraction,
    {
        "priority_classification": "priority_classification",
        "final_report": "final_report"
    }
)

workflow.add_edge("priority_classification", "final_report")

workflow.add_edge("final_report", END)


# Compile LangGraph workflow
meeting_graph = workflow.compile()


# Display graph configuration
print("=" * 70)
print("LANGGRAPH WORKFLOW CONFIGURATION")
print("=" * 70)

print(f"State Schema     : {MeetingState.__name__}")
print(f"Registered Nodes : {len(workflow.nodes)}")
print(f"Graph Compiled   : {meeting_graph is not None}")

print("\nRegistered Nodes:")
print("-" * 70)

for node_name in workflow.nodes:
    print(f"- {node_name}")

print("\nConditional Routing:")
print("-" * 70)
print("Action items present : Priority Classification -> Final Report")
print("No action items      : Final Report (skip priority agent)")

print("\n" + "=" * 70)
print("Status: LangGraph workflow compiled successfully.")

LANGGRAPH WORKFLOW CONFIGURATION
State Schema     : MeetingState
Registered Nodes : 5
Graph Compiled   : True

Registered Nodes:
----------------------------------------------------------------------
- topic_extraction
- meeting_summary
- action_extraction
- priority_classification
- final_report

Conditional Routing:
----------------------------------------------------------------------
Action items present : Priority Classification -> Final Report
No action items      : Final Report (skip priority agent)

Status: LangGraph workflow compiled successfully.


## Step 4.8: Develop the Final Report Generator

**Objective:**

Develop a reusable report generation function that consolidates all LangGraph agent outputs into a structured meeting analysis report.

**Report Components:**

1. Meeting Summary
2. Key Discussion Topics
3. Action Items with Owners and Deadlines
4. Task Priority Classification
5. Meeting Analysis Statistics

**Implementation Approach:**

- Read the outputs from MeetingState.
- Generate a Markdown-formatted report.
- Match each action item with its priority classification.
- Display unassigned owners as "Not specified".
- Normalize missing deadlines.
- Calculate priority distribution.
- Handle meetings with no action items.
- Return the report through the final_report state field.

**Design Considerations:**

- No additional LLM calls.
- Consistent report formatting.
- No invented meeting information.
- Support conditional workflow execution.
- Reusable for different meeting transcripts.

**Expected Outcome:**

A complete Markdown meeting analysis report ready for end-to-end LangGraph execution and eventual Streamlit integration.

In [24]:
# Step 4.8: Develop the Final Report Generator

from collections import Counter


def final_report_node(state: MeetingState) -> dict:
    """Generate a Markdown meeting analysis report."""

    summary = state.get("summary", "Summary not available.")
    topics = state.get("topics", [])
    actions = state.get("action_items", [])
    priorities = state.get("priorities", [])

    # Map task descriptions to their classifications
    priority_map = {
        item.task: item.priority
        for item in priorities
    }

    # Normalize missing values
    def clean_value(value):
        if value is None:
            return "Not specified"

        text = str(value).strip()

        if text.lower() in {
            "", "null", "none", "n/a", "unknown"
        }:
            return "Not specified"

        return text

    # Build report
    report = [
        "# AI Meeting Analysis Report",
        "",
        "## 1. Meeting Summary",
        "",
        summary,
        "",
        "## 2. Key Discussion Topics",
        ""
    ]

    if topics:
        for index, topic in enumerate(topics, start=1):
            report.append(f"{index}. {topic}")
    else:
        report.append("No discussion topics identified.")

    report.extend([
        "",
        "## 3. Action Items",
        ""
    ])

    if actions:
        report.append(
            "| # | Action Item | Owner | Deadline | Priority |"
        )
        report.append(
            "|---|---|---|---|---|"
        )

        for index, action in enumerate(actions, start=1):
            priority = priority_map.get(
                action.task,
                "Not classified"
            )

            # Escape Markdown table separators
            task = action.task.replace("|", r"\|")
            owner = clean_value(action.owner).replace("|", r"\|")
            deadline = clean_value(action.deadline).replace("|", r"\|")

            report.append(
                f"| {index} | {task} | {owner} | "
                f"{deadline} | {priority} |"
            )

    else:
        report.append("No action items identified.")

    # Priority distribution
    counts = Counter(
        item.priority for item in priorities
    )

    report.extend([
        "",
        "## 4. Priority Distribution",
        "",
        f"- High Priority: {counts.get('High', 0)}",
        f"- Medium Priority: {counts.get('Medium', 0)}",
        f"- Low Priority: {counts.get('Low', 0)}",
        "",
        "## 5. Meeting Analysis Statistics",
        "",
        f"- Total Discussion Topics: {len(topics)}",
        f"- Total Action Items: {len(actions)}",
        f"- Classified Tasks: {len(priorities)}",
        f"- Unassigned Tasks: {sum(clean_value(a.owner) == 'Not specified' for a in actions)}",
    ])

    return {
        "final_report": "\n".join(report)
    }


# Test the report generator with existing validated results
report_test_state = {
    **validated_state,
    "priorities": classified_priorities
}

report_result = final_report_node(report_test_state)

generated_report = report_result["final_report"]

# Display formatted report in Jupyter
from IPython.display import display, Markdown

display(Markdown(generated_report))

print("\n" + "=" * 70)
print("FINAL REPORT GENERATOR - VALIDATION")
print("=" * 70)

print(f"Report Generated  : {bool(generated_report.strip())}")
print(f"Report Characters : {len(generated_report):,}")
print(f"Discussion Topics : {len(report_test_state['topics'])}")
print(f"Action Items      : {len(report_test_state['action_items'])}")
print(f"Classified Tasks  : {len(report_test_state['priorities'])}")

print("=" * 70)
print("Status: Final Report Generator executed successfully.")

# AI Meeting Analysis Report

## 1. Meeting Summary

The meeting addressed urgent complaints regarding the mobile app crashing during login, particularly affecting Android users. The team identified that the issue may be linked to a recent update to the authentication API deployed three days ago, coinciding with the spike in crash reports. Action items include the Mobile Developer reviewing the login module, the QA Tester preparing a detailed bug report, and the Backend Developer verifying the API response format. The team agreed to prioritize these investigations and communicate updates to both management and affected users. A follow-up meeting is scheduled for tomorrow morning to assess progress.

## 2. Key Discussion Topics

1. Mobile App Crashing Issues
2. Root Cause Analysis of Login Failures
3. Impact of Recent Backend Changes
4. Testing on Different Android Versions
5. Communication Strategy for Affected Users
6. Error Handling Improvements in Mobile App
7. Documentation and Issue Tracking
8. Urgent Resolution and Patch Deployment

## 3. Action Items

| # | Action Item | Owner | Deadline | Priority |
|---|---|---|---|---|
| 1 | Review the Android login module and check if any error handling is missing. | Mobile Developer | Not specified | High |
| 2 | Prepare and share the detailed bug report with screenshots and logs. | QA Tester | Within the next hour | High |
| 3 | Verify the authentication API and check if there are any breaking changes. | Backend Developer | Not specified | High |
| 4 | Test the login flow on Android 11, 12, and 13. | QA Tester | Not specified | Medium |
| 5 | Review server and API logs for authentication errors. | Backend Developer | Right away | High |
| 6 | Send a message to the support team to update customers about the fix. | Customer Support Lead | Not specified | Medium |
| 7 | Begin working on the login fix immediately; if the issue is in the login module, aim to push the fix by tomorrow. | Mobile Developer | Start immediately after the meeting; conditional fix target: by tomorrow | High |
| 8 | Perform regression testing once the fix is ready. | QA Tester | Not specified | Medium |
| 9 | Test the authentication endpoints after the fix is deployed. | Backend Developer | Not specified | Medium |
| 10 | Document the issue and the resolution steps. | Not specified | Not specified | Low |
| 11 | Update the issue tracker with all findings. | QA Tester | Not specified | Medium |
| 12 | Prepare a communication message for users once the fix is deployed. | Not specified | Not specified | Low |
| 13 | Add better error handling to prevent app crashes due to API failures. | Mobile Developer | Not specified | Medium |
| 14 | Release the patch update to the Play Store after testing confirms the issue is resolved. | Not specified | After successful testing and confirmation | Medium |
| 15 | Provide an update by the end of the day. | Mobile Developer | By the end of the day | Medium |
| 16 | Report any issues found during the review of the API logs. | Backend Developer | Not specified | Medium |
| 17 | Regroup tomorrow morning to review the progress. | Not specified | Tomorrow morning | Low |

## 4. Priority Distribution

- High Priority: 5
- Medium Priority: 9
- Low Priority: 3

## 5. Meeting Analysis Statistics

- Total Discussion Topics: 8
- Total Action Items: 17
- Classified Tasks: 17
- Unassigned Tasks: 4


FINAL REPORT GENERATOR - VALIDATION
Report Generated  : True
Report Characters : 3,394
Discussion Topics : 8
Action Items      : 17
Classified Tasks  : 17
Status: Final Report Generator executed successfully.


## Step 4.9: End-to-End LangGraph Workflow Execution

**Objective:**

Execute the complete multi-agent LangGraph workflow using the original meeting transcript.

**Workflow Execution:**

1. Initialize the graph with the meeting transcript.
2. Execute the Topic Extraction Agent.
3. Execute the Meeting Summary Agent.
4. Execute the Action Item Extraction Agent.
5. Evaluate conditional routing.
6. Execute the Priority Classification Agent when action items exist.
7. Generate the final meeting analysis report.
8. Return the completed LangGraph state.

**Validation Criteria:**

- Successful graph compilation and execution.
- Topic extraction completed.
- Meeting summary generated.
- Action items extracted.
- Priority classifications match extracted action items.
- Final report generated.
- No unexpected workflow errors.

**Evaluation Considerations:**

- Total execution time.
- Number of topics extracted.
- Number of action items identified.
- Priority distribution.
- Completeness of final state.
- Differences from the manually reconciled reference.

**Expected Outcome:**

A fully executed LangGraph workflow producing a complete AI Meeting Analysis Report.

In [25]:
# Step 4.9: End-to-End LangGraph Workflow Execution

import os
import time
from collections import Counter
from langgraph.graph import StateGraph, START, END
from IPython.display import display, Markdown

# Apply previously identified SSL environment fix
os.environ.pop("SSL_CERT_FILE", None)

# Rebuild graph to include the updated final_report_node
production_workflow = StateGraph(MeetingState)

# Register all specialized agents
production_workflow.add_node(
    "topic_extraction",
    topic_extraction_agent
)

production_workflow.add_node(
    "meeting_summary",
    meeting_summary_agent
)

production_workflow.add_node(
    "action_extraction",
    refined_action_extraction_agent
)

production_workflow.add_node(
    "priority_classification",
    priority_classification_agent
)

production_workflow.add_node(
    "final_report",
    final_report_node
)

# Define sequential transitions
production_workflow.add_edge(
    START,
    "topic_extraction"
)

production_workflow.add_edge(
    "topic_extraction",
    "meeting_summary"
)

production_workflow.add_edge(
    "meeting_summary",
    "action_extraction"
)

# Conditional routing
production_workflow.add_conditional_edges(
    "action_extraction",
    route_after_action_extraction,
    {
        "priority_classification": "priority_classification",
        "final_report": "final_report"
    }
)

production_workflow.add_edge(
    "priority_classification",
    "final_report"
)

production_workflow.add_edge(
    "final_report",
    END
)

# Compile updated workflow
meeting_graph = production_workflow.compile()

# Execute workflow with original transcript
start_time = time.perf_counter()

final_state = meeting_graph.invoke({
    "transcript": cleaned_transcript
})

execution_time = time.perf_counter() - start_time

# Extract workflow outputs
workflow_topics = final_state.get("topics", [])
workflow_summary = final_state.get("summary", "")
workflow_actions = final_state.get("action_items", [])
workflow_priorities = final_state.get("priorities", [])
workflow_report = final_state.get("final_report", "")

# Calculate priority distribution
workflow_priority_counts = Counter(
    item.priority for item in workflow_priorities
)

# Validate workflow outputs
assert workflow_topics, "Topic extraction failed."
assert workflow_summary.strip(), "Meeting summary is missing."
assert workflow_report.strip(), "Final report is missing."

assert len(workflow_actions) == len(workflow_priorities), (
    "Action items and priority classifications do not match."
)

# Display execution results
print("=" * 75)
print("END-TO-END LANGGRAPH EXECUTION RESULTS")
print("=" * 75)

print(f"Graph Compiled       : {meeting_graph is not None}")
print(f"Execution Time       : {execution_time:.2f} seconds")
print(f"Topics Extracted     : {len(workflow_topics)}")
print(f"Action Items         : {len(workflow_actions)}")
print(f"Classified Tasks     : {len(workflow_priorities)}")
print(f"Final Report Created : {bool(workflow_report.strip())}")

print("\nPRIORITY DISTRIBUTION")
print("-" * 75)

for level in ["High", "Medium", "Low"]:
    print(
        f"{level:<8}: "
        f"{workflow_priority_counts.get(level, 0)}"
    )

print("\nREFERENCE COMPARISON")
print("-" * 75)

print(f"Reference Action Items : {len(final_actions)}")
print(f"Workflow Action Items  : {len(workflow_actions)}")
print(
    f"Count Difference       : "
    f"{len(workflow_actions) - len(final_actions):+d}"
)

print("\n" + "=" * 75)
print("Status: End-to-end LangGraph execution completed.")
print("=" * 75)

# Display the final report
print("\nGENERATED MEETING ANALYSIS REPORT\n")

display(Markdown(workflow_report))

END-TO-END LANGGRAPH EXECUTION RESULTS
Graph Compiled       : True
Execution Time       : 16.33 seconds
Topics Extracted     : 8
Action Items         : 17
Classified Tasks     : 17
Final Report Created : True

PRIORITY DISTRIBUTION
---------------------------------------------------------------------------
High    : 4
Medium  : 12
Low     : 1

REFERENCE COMPARISON
---------------------------------------------------------------------------
Reference Action Items : 17
Workflow Action Items  : 17
Count Difference       : +0

Status: End-to-end LangGraph execution completed.

GENERATED MEETING ANALYSIS REPORT



# AI Meeting Analysis Report

## 1. Meeting Summary

The meeting addressed urgent complaints regarding the mobile app crashing during login, particularly affecting Android users. The team identified that the issue may be linked to a recent update to the authentication API deployed three days ago, coinciding with the spike in crash reports. Action items include the Mobile Developer reviewing the login module, the QA Tester preparing a detailed bug report, and the Backend Developer verifying the API response format. The team agreed to prioritize these tasks and regroup the following morning to assess progress, while also preparing communication for affected users.

## 2. Key Discussion Topics

1. Mobile App Crashing Issues
2. Root Cause Analysis of Login Failures
3. Impact of Recent Backend Changes
4. Testing on Different Android Versions
5. Communication Strategy for Affected Users
6. Error Handling Improvements in Mobile App
7. Documentation and Issue Tracking
8. Patch Update Release Process

## 3. Action Items

| # | Action Item | Owner | Deadline | Priority |
|---|---|---|---|---|
| 1 | Review the Android login module and check if any error handling is missing. | Mobile Developer | by tomorrow. | High |
| 2 | Prepare a detailed bug report with screenshots and logs. | QA Tester | Not specified | Medium |
| 3 | Verify the authentication API and check if there are any breaking changes. | Backend Developer | Not specified | Medium |
| 4 | Test the login flow on Android 11, 12, and 13. | QA Tester | Not specified | Medium |
| 5 | Check the server logs to see if any authentication errors are recorded. | Backend Developer | Not specified | Medium |
| 6 | Send a message to the support team so they can update customers that we are working on a fix. | Customer Support Lead | Not specified | Medium |
| 7 | Document the issue and the resolution steps. | Not specified | Not specified | Medium |
| 8 | Update the issue tracker with all findings. | QA Tester | Not specified | Medium |
| 9 | Notify users once the fix is deployed. | Not specified | Not specified | Medium |
| 10 | Add better error handling so the app does not crash even if the API fails. | Mobile Developer | Not specified | Medium |
| 11 | Release the patch update to the Play Store after testing. | QA Tester | Not specified | Medium |
| 12 | Start working on the fix immediately after this meeting. | Mobile Developer | immediately after this meeting | High |
| 13 | Review the API logs right away. | Backend Developer | right away | High |
| 14 | Share the bug report within the next hour. | QA Tester | within the next hour | High |
| 15 | Provide an update by the end of the day. | Mobile Developer | by the end of the day | Medium |
| 16 | Report any issues I find. | Backend Developer | by the end of the day | Medium |
| 17 | Regroup tomorrow morning to review the progress. | Not specified | tomorrow morning | Low |

## 4. Priority Distribution

- High Priority: 4
- Medium Priority: 12
- Low Priority: 1

## 5. Meeting Analysis Statistics

- Total Discussion Topics: 8
- Total Action Items: 17
- Classified Tasks: 17
- Unassigned Tasks: 3